# AgroScan AI — Notebook 01: PlantVillage Dataset EDA

This notebook performs exploratory data analysis of the PlantVillage dataset:

1. Class distribution across 38 disease categories
2. Sample images per disease class
3. Augmentation demonstration
4. Class imbalance analysis
5. Data split strategy

> **Dataset note:** The notebook uses the PlantVillage dataset. If not downloaded,
> synthetic placeholder images are used for demonstration.
> See `data/README.md` for download instructions.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path
from PIL import Image

sys.path.insert(0, '..')
np.random.seed(42)
print('Libraries loaded.')

## 1. PlantVillage Class Overview

In [ ]:
# PlantVillage 38-class breakdown (approximate counts from public release)
plantvillage_classes = [
    ('Apple', 'Apple_scab', 2016),
    ('Apple', 'Black_rot', 1987),
    ('Apple', 'Cedar_apple_rust', 1760),
    ('Apple', 'healthy', 1645),
    ('Blueberry', 'healthy', 1502),
    ('Cherry', 'Powdery_mildew', 1683),
    ('Cherry', 'healthy', 854),
    ('Corn', 'Cercospora_leaf_spot_Gray_leaf_spot', 513),
    ('Corn', 'Common_rust', 1192),
    ('Corn', 'Northern_Leaf_Blight', 985),
    ('Corn', 'healthy', 1162),
    ('Grape', 'Black_rot', 1180),
    ('Grape', 'Esca_Black_Measles', 1383),
    ('Grape', 'Leaf_blight_Isariopsis_Leaf_Spot', 1076),
    ('Grape', 'healthy', 423),
    ('Orange', 'Haunglongbing_Citrus_greening', 5507),
    ('Peach', 'Bacterial_spot', 2297),
    ('Peach', 'healthy', 360),
    ('Pepper_bell', 'Bacterial_spot', 997),
    ('Pepper_bell', 'healthy', 1478),
    ('Potato', 'Early_blight', 1000),
    ('Potato', 'Late_blight', 1000),
    ('Potato', 'healthy', 152),
    ('Raspberry', 'healthy', 371),
    ('Soybean', 'healthy', 5090),
    ('Squash', 'Powdery_mildew', 1835),
    ('Strawberry', 'Leaf_scorch', 1109),
    ('Strawberry', 'healthy', 456),
    ('Tomato', 'Bacterial_spot', 2127),
    ('Tomato', 'Early_blight', 1000),
    ('Tomato', 'Late_blight', 1909),
    ('Tomato', 'Leaf_Mold', 952),
    ('Tomato', 'Septoria_leaf_spot', 1771),
    ('Tomato', 'Spider_mites_Two_spotted_spider_mite', 1676),
    ('Tomato', 'Target_Spot', 1404),
    ('Tomato', 'Tomato_Yellow_Leaf_Curl_Virus', 5357),
    ('Tomato', 'Tomato_mosaic_virus', 373),
    ('Tomato', 'healthy', 1591),
]

crops = [c[0] for c in plantvillage_classes]
conditions = [c[1] for c in plantvillage_classes]
counts = [c[2] for c in plantvillage_classes]
labels = [f"{c[0]}\n{c[1][:20]}" for c in plantvillage_classes]

print(f'Total classes: {len(plantvillage_classes)}')
print(f'Total images: {sum(counts):,}')
print(f'Unique crops: {len(set(crops))}')
print(f'Min class size: {min(counts)} ({conditions[counts.index(min(counts))]})')
print(f'Max class size: {max(counts)} ({conditions[counts.index(max(counts))]})')
print(f'Mean class size: {np.mean(counts):.0f}')

## 2. Class Distribution Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
x_pos = np.arange(len(counts))

# Color by crop
unique_crops = list(dict.fromkeys(crops))
crop_colors = plt.cm.Set3(np.linspace(0, 1, len(unique_crops)))
color_map = {crop: crop_colors[i] for i, crop in enumerate(unique_crops)}
bar_colors = [color_map[c] for c in crops]

bars = ax.bar(x_pos, counts, color=bar_colors, edgecolor='white', linewidth=0.5)

ax.set_xticks(x_pos)
short_labels = [f"{c[0][:5]}_{c[1][:12]}" for c in plantvillage_classes]
ax.set_xticklabels(short_labels, rotation=75, ha='right', fontsize=7)

# Legend for crops
legend_patches = [mpatches.Patch(color=color_map[c], label=c) for c in unique_crops]
ax.legend(handles=legend_patches, loc='upper right', ncol=2, fontsize=8, title='Crop')

ax.axhline(y=np.mean(counts), color='red', linestyle='--', linewidth=1.5, label=f'Mean ({np.mean(counts):.0f})')
ax.set_title('PlantVillage — Sample Count per Class (38 classes)', fontsize=13)
ax.set_ylabel('Number of images')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Healthy vs Diseased Distribution per Crop

In [ ]:
crop_healthy = {}
crop_diseased = {}

for crop, condition, count in plantvillage_classes:
    if 'healthy' in condition:
        crop_healthy[crop] = crop_healthy.get(crop, 0) + count
    else:
        crop_diseased[crop] = crop_diseased.get(crop, 0) + count

all_crops = sorted(set(list(crop_healthy.keys()) + list(crop_diseased.keys())))
h_counts = [crop_healthy.get(c, 0) for c in all_crops]
d_counts = [crop_diseased.get(c, 0) for c in all_crops]

x = np.arange(len(all_crops))
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 0.2, h_counts, 0.4, label='Healthy', color='mediumseagreen')
ax.bar(x + 0.2, d_counts, 0.4, label='Diseased', color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(all_crops, rotation=45, ha='right')
ax.set_title('Healthy vs Diseased Sample Count per Crop', fontsize=12)
ax.set_ylabel('Images')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Crops with fewest healthy samples: Potato (152), Grape (423), Peach (360)')
print('These crops benefit most from weighted sampling or data augmentation.')

## 4. Synthetic Leaf Image Generation

In [ ]:
def generate_synthetic_leaf(condition='healthy', size=224):
    """Generate a synthetic leaf image for demonstration."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    # Background
    img[:] = (200, 220, 200)
    # Leaf shape
    pts = np.array([
        [size//2, 20], [size-20, size//2], [size//2, size-20], [20, size//2]
    ], dtype=np.int32)
    cv2.fillPoly(img, [pts], (34, 139, 34))
    # Vein
    cv2.line(img, (size//2, 20), (size//2, size-20), (25, 110, 25), 2)
    cv2.line(img, (size//2, size//2-30), (size-30, size//2+20), (25, 110, 25), 1)
    cv2.line(img, (size//2, size//2-30), (30, size//2+20), (25, 110, 25), 1)

    if condition == 'early_blight':
        for _ in range(5):
            cx = np.random.randint(50, 174)
            cy = np.random.randint(50, 174)
            r1, r2 = np.random.randint(8, 18), np.random.randint(5, 12)
            cv2.ellipse(img, (cx, cy), (r1, r2), 0, 0, 360, (101, 67, 33), -1)
            cv2.ellipse(img, (cx, cy), (r1+3, r2+3), 0, 0, 360, (150, 100, 50), 2)
    elif condition == 'late_blight':
        for _ in range(3):
            pts_d = np.random.randint(60, 164, (8, 2)).astype(np.int32)
            cv2.fillPoly(img, [pts_d], (40, 50, 30))
    elif condition == 'powdery_mildew':
        for _ in range(40):
            cx, cy = np.random.randint(40, 184, 2)
            cv2.circle(img, (cx, cy), np.random.randint(3, 8), (200, 200, 180), -1)
    elif condition == 'bacterial_spot':
        for _ in range(15):
            cx, cy = np.random.randint(40, 184, 2)
            cv2.circle(img, (cx, cy), np.random.randint(4, 10), (90, 60, 40), -1)
            cv2.circle(img, (cx, cy), np.random.randint(4, 10)+3, (120, 90, 60), 1)

    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


conditions_demo = ['healthy', 'early_blight', 'late_blight', 'powdery_mildew', 'bacterial_spot']
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Synthetic Leaf Images by Disease Condition', fontsize=13)

for col, cond in enumerate(conditions_demo):
    for row in range(2):
        img = generate_synthetic_leaf(cond)
        axes[row, col].imshow(img)
        axes[row, col].set_title(cond.replace('_', ' '), fontsize=8)
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 5. Augmentation Demonstration

In [ ]:
try:
    from agroscan.data.transforms import get_train_transforms, get_val_transforms
    import torch

    train_tf = get_train_transforms(224)
    val_tf = get_val_transforms(224)

    leaf_img = generate_synthetic_leaf('early_blight', size=224)
    pil_img = Image.fromarray(leaf_img)

    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    fig.suptitle('Training Augmentations Applied to a Single Leaf Image', fontsize=12)

    # Row 0: original + 4 augmented
    axes[0, 0].imshow(leaf_img)
    axes[0, 0].set_title('Original (val transform)', fontsize=9)
    axes[0, 0].axis('off')

    for i in range(1, 5):
        aug_tensor = train_tf(pil_img)
        if isinstance(aug_tensor, torch.Tensor):
            aug_np = aug_tensor.permute(1,2,0).numpy()
            aug_np = ((aug_np - aug_np.min()) / (aug_np.max() - aug_np.min() + 1e-8) * 255).astype(np.uint8)
        else:
            aug_np = np.array(aug_tensor)
        axes[0, i].imshow(aug_np)
        axes[0, i].set_title(f'Aug #{i}', fontsize=9)
        axes[0, i].axis('off')

    # Row 1: pixel stats
    for i, cond in enumerate(conditions_demo):
        img = generate_synthetic_leaf(cond)
        aug = train_tf(Image.fromarray(img))
        if isinstance(aug, torch.Tensor):
            aug_np = aug.permute(1,2,0).numpy()
        else:
            aug_np = np.array(aug) / 255.0
        axes[1, i].hist(aug_np.flatten(), bins=50, color='green', alpha=0.7)
        axes[1, i].set_title(f'{cond}\npixel dist.', fontsize=8)
        axes[1, i].set_xlim(-3, 3)

    plt.tight_layout()
    plt.show()

except ImportError as e:
    print(f'AgroScan transforms not available: {e}')
    print('Run: pip install -e .. from project root.')

## 6. Recommended Train/Val Split Strategy

In [ ]:
total = sum(counts)
train_size = int(total * 0.8)
val_size = total - train_size

print('=== Recommended Data Split ===')
print(f'Total images:   {total:,}')
print(f'Train (80%):    {train_size:,}')
print(f'Val (20%):      {val_size:,}')
print()
print('Recommendations:')
print('  - Stratified split to preserve per-class ratios')
print('  - WeightedRandomSampler for underrepresented classes')
print('  - Aggressive augmentation for classes with < 500 samples')
print(f'  - Classes with < 500 samples: {sum(1 for c in counts if c < 500)} classes')

fig, ax = plt.subplots(figsize=(8, 4))
ax.pie([train_size, val_size],
       labels=[f'Train\n{train_size:,} (80%)', f'Val\n{val_size:,} (20%)'],
       colors=['steelblue', 'orange'],
       autopct='%1.1f%%', startangle=90)
ax.set_title('PlantVillage Train/Val Split', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

| Statistic | Value |
|-----------|-------|
| Total images | ~87,848 |
| Classes | 38 |
| Unique crops | 14 |
| Min class | 152 (Potato healthy) |
| Max class | 5,507 (Orange citrus greening) |
| Class imbalance ratio | ~36× |

**Next:** Notebook 02 — Grad-CAM visualisation for explaining model predictions.